In [3]:
print(1234)

1234


In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from openai import OpenAI
openai_client = OpenAI()

In [7]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [8]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s going on?'

In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [48]:
question ="I just discovered the course. Can I still join?"
answer = llm(question)
print(answer)

Yes — in most cases you can still join, but it depends on the course’s enrollment rules and whether it’s already started.

If you want, I can help you figure it out quickly. Usually the key things are:
- **Whether enrollment is still open**
- **The course start date**
- **Whether there’s a late registration option**
- **Any prerequisites or waitlist**

If you’re asking about a specific course, send me:
- the **course name**
- the **school/platform**
- and, if you know it, **when it started**

I can help you draft a message to the instructor or check what to do next.


In [49]:
prompt = f"""
Your task is to answer questions from the course participants based on the provided context. 
Use the information in the context to provide accurate and helpful responses. If the question is not answerable based on the context, 
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [50]:
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [28]:
question ="GPU options?"

In [29]:
prompt = f"""
Your task is to answer questions from the course participants based on the provided context. 
Use the information in the context to provide accurate and helpful responses. If the question is not answerable based on the context, 
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [30]:

answer = llm(prompt)
print(answer)

Potential GPU options include **Google Colab, Kaggle, and Databricks**.  
Also, be sure to **check the quota and reset cycle carefully**.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [31]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()


In [32]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [36]:
documents[1100]

{'id': '841966c903',
 'course': 'mlops-zoomcamp',
 'section': 'Module 3: Orchestration',
 'question': 'Where is the FAQ for Prefect questions?',
 'answer': '[Here](https://docs.google.com/document/d/1Nyktf7WoRec5lDUBREXL5zLI1Edbw9_R8e45fDn4KB8/edit?usp=sharing)'}

In [37]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [58]:
search_results =index.search(question,
                             boost_dict={"question": 2.0, "answer": 1.0},
                             filter_dict={"course": "llm-zoomcamp"},
                             num_results=5)



In [51]:
def search(question,course="llm-zoomcamp"):
    boost_dict={"question": 2.0, "section": 0.5}
    filter_dict={"course": course}
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict= filter_dict,
        num_results=5)

In [ ]:
search(question)

In [53]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [ ]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}

"""


In [56]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [61]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

In [62]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced m

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()